# Logistic Regression From Scratch — Part II

## Introduction

In Part I, we implemented Logistic Regression from scratch using NumPy. We learned how to calculate probabilities using the sigmoid function, calculate binary cross-entropy loss, and train the model using gradient descent.

In Part II, we focus on testing and evaluating our previously trained model.

Instead of training the model again, we load the saved weights and bias from Part I and use them to make predictions and evaluate classification performance.

### Objectives
- Load a previously trained Logistic Regression model.
- Recreate prediction functions.
- Test the model on new data.
- Understand the confusion matrix.
- Implement accuracy, precision, and recall using NumPy.
- Evaluate model performance.

## 1. Loading the Saved Model

In Part I, we saved the trained model parameters (weight and bias) in an `.npz` file.

We load these parameters in Part II using NumPy's `np.load()` function.

- **Weight (w):** The learned coefficient used in the linear combination.
- **Bias (b):** The learned intercept added to the linear combination.

By loading these parameters, we can reuse the trained model without performing training again.

In [1]:
import numpy as np
saved_model = np.load("model/logistic_regression_part_I_model.npz")

w_loaded = saved_model["w"].item()
b_loaded = saved_model["b"].item()

print("Weight :",w_loaded)
print("Bias :",b_loaded)

Weight : 0.40808303784853656
Bias : -1.4263280576885489


## 2. Recreating Prediction Functions

After loading the model parameters, we recreate the prediction functions.

### Sigmoid Function

The sigmoid function converts a linear output into a probability between 0 and 1.

\[
sigma(z)= 1/(1+np.exp(-z))
\]

### Probability Prediction

First, we calculate the linear combination:

\[
z=wx+b
\]

Then we apply the sigmoid function to obtain the probability.

### Class Prediction

We use a threshold of 0.5 to convert probabilities into binary classes.

- Probability greater than or equal to 0.5: Class 1.
- Probability less than 0.5: Class 0.

In [10]:
def sigmoid(z):  # Sigmoid function
    return 1/(1+np.exp(-z))

def predict_probability(x,w,b):
    z = x*w+b
    pred_prob = sigmoid(z)
    return pred_prob

def predict_class(x,w,b):
    p = predict_probability(x,w,b)
    return (p >= 0.5).astype(bool)
def log_loss(y,p):
    n = len(y)
    loss = -(1/n) * np.sum(
        y * np.log(p) + (1-y) * np.log(1-p)
    )
print("Done")


Done


### Loading the same data we where used in part-I

In [11]:
X = np.array([1,2,3,4,5,6,7,8,9])
y = np.array([0,0,0,0,1,1,1,1,1])

print(f"X shape : {X.shape} and Y shape : {y.shape}")

X shape : (9,) and Y shape : (9,)


### Applying the prediction functions on dataset

In [12]:
z = predict_probability(X,w_loaded,b_loaded)
pred_class = predict_class(X,w_loaded,b_loaded)

print("Predict Claases :",pred_class)

Predict Claases : [False False False  True  True  True  True  True  True]


## 3. Testing on Unseen Data

After loading the trained model, we test it using new inputs that were not used during training.

The model calculates:
- **Predicted probability:** The estimated probability of belonging to class 1.
- **Predicted class:** The final binary classification based on the 0.5 threshold.

This helps us observe how the trained model behaves on new inputs.

In [15]:
new_data = np.array([4.5,6.7,2.8])
new_pred = predict_probability(new_data,w_loaded,b_loaded)
new_pred_class = predict_class(new_data,w_loaded,b_loaded)

print("New prediction Class :",new_pred_class)

New prediction Class : [ True  True False]


## 4. Preparing the Evaluation Dataset

To evaluate a classification model, we need actual labels to compare against predicted labels.

We prepare a separate test dataset containing:
- `X_test`: Input features.
- `y_test`: Actual binary labels.

The model predictions are compared with these actual labels to calculate evaluation metrics.

**Note:** This is a small educational dataset used to understand evaluation calculations. It is not sufficient for reliable real-world performance assessment.

In [17]:
X_test = np.array([1.5, 3.5, 4.5, 5.5, 6.5])
y_test = np.array([0, 0, 0, 1, 1])
X_test.shape, y_test.shape

((5,), (5,))

## 5. Confusion Matrix

A confusion matrix is used to evaluate a classification model by comparing actual labels with predicted labels.

It contains four outcomes:

| Term | Meaning |
|---|---|
| True Positive (TP) | Actual = 1, Predicted = 1 |
| True Negative (TN) | Actual = 0, Predicted = 0 |
| False Positive (FP) | Actual = 0, Predicted = 1 |
| False Negative (FN) | Actual = 1, Predicted = 0 |

### Matrix Structure

\[
CM=
\begin{bmatrix}
TN & FP\\
FN & TP
\end{bmatrix}
\]

- Rows represent actual classes.
- Columns represent predicted classes.

### Why this arrangement?

The first row represents actual class 0, so it contains TN and FP.

The second row represents actual class 1, so it contains FN and TP.

The confusion matrix helps us understand which predictions were correct and which were incorrect.

In [25]:
def confusion_matrix(y_actual, y_predicted):

    tn = np.sum((y_actual == 0) & (y_predicted == 0))
    fp = np.sum((y_actual == 0) & (y_predicted == 1))
    fn = np.sum((y_actual == 1) & (y_predicted == 0))
    tp = np.sum((y_actual == 1) & (y_predicted == 1))

    accuracy = (tp + tn) / (tp + tn + fp + fn)

    recall = tp / (tp + fn) if (tp + fn) != 0 else 0

    precision = tp / (tp + fp) if (tp + fp) != 0 else 0

    cm = np.array([
        [tn, fp],
        [fn, tp]
    ])

    return cm, accuracy, recall, precision
print("Done")

Done


## 6. Accuracy

Accuracy measures how many predictions the model classified correctly out of all predictions.

### Formula

\[
Accuracy = {TP+TN}/{TP+TN+FP+FN}
\]

A higher accuracy means the model correctly classified a larger proportion of the evaluation examples.

However, accuracy alone may not be sufficient when classes are imbalanced.

## 7. Precision

Precision measures how many predicted positive cases were actually positive.

In simple terms, it tells us how reliable the model's positive predictions are.

### Formula

\[
Precision={TP}/{TP+FP}
\]

A higher precision means fewer false positive predictions relative to all positive predictions.

If the denominator is zero, precision is undefined; our implementation returns 0 as a practical convention.

## 8. Recall

Recall measures how many actual positive cases the model correctly identified.

In simple terms, it tells us how effectively the model finds positive cases.

### Formula

\[
Recall={TP}/{TP+FN}
\]

A higher recall means fewer actual positive cases were missed by the model.

If the denominator is zero, recall is undefined; our implementation returns 0 as a practical convention.

In [26]:
y_pred = predict_class(X_test, w_loaded, b_loaded)
cm, accuracy, recall, precision = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:\n", cm)
print("Accuracy:", accuracy)
print("Recall:", recall)
print("Precision:", precision)

Confusion Matrix:
 [[1 2]
 [0 2]]
Accuracy: 0.6
Recall: 1.0
Precision: 0.5


## 9. Final Model Evaluation

After implementing the confusion matrix, accuracy, precision, and recall using NumPy, we evaluated our trained Logistic Regression model.

### Evaluation Results

| Metric | Result |
|---|---:|
| Accuracy | 0.60 (60%) |
| Recall | 1.00 (100%) |
| Precision | 0.50 (50%) |

### Confusion Matrix

\[
CM=
\begin{bmatrix}
1 & 2\\
0 & 2
\end{bmatrix}
\]

### Observations

- The model correctly classified 3 out of 5 test examples.
- Recall was 100%, meaning all actual positive cases were identified.
- Precision was 50%, meaning only half of the positive predictions were correct.
- The model produced two false positive predictions.

### Conclusion

In Part II, we successfully loaded the previously trained Logistic Regression model and evaluated it using a separate test dataset.

We implemented the confusion matrix, accuracy, precision, and recall from scratch using NumPy without relying on Scikit-learn's evaluation functions.

This completed our basic Logistic Regression implementation and evaluation workflow.